In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
file_path = "/content/drive/MyDrive/Titanic_Dataset.csv"

In [4]:
import pandas as pd
df = pd.read_csv(file_path)

In [5]:
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
trf1 = ColumnTransformer(
    [('impute_age', SimpleImputer(strategy='median'), ['Age'])],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

In [9]:
from sklearn.base import BaseEstimator, TransformerMixin
class FamilyAndNameEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # 1. Family Features
        X['FamilySize'] = X['SibSp'] + X['Parch'] + 1
        X['IsAlone'] = (X['FamilySize'] == 1).astype(int)

        # 2. Child Feature
        X['Child'] = (X['Age'] < 16).astype(int)

        # 3. Name Features
        X['Title'] = X['Name'].str.extract(r',\s*([^.]*)\.')
        X['Surname'] = X['Name'].str.split(',').str[0]

        # 4. Map Sex
        X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})

        return X

    def get_feature_names_out(self, input_features=None):
        return np.array(['SibSp', 'Parch', 'Age', 'Name', 'Sex',
                         'FamilySize', 'IsAlone', 'Child', 'Title', 'Surname'], dtype='object')

trf2 = ColumnTransformer(
    [('family_name_eng', FamilyAndNameEngineer(), ['SibSp', 'Parch', 'Age', 'Name', 'Sex'])],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

In [10]:
class TicketEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.ticket_sizes_ = X.groupby('Ticket').size().to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        # 1. Ticket Group Size
        X['TicketGroupSize'] = X['Ticket'].map(self.ticket_sizes_).fillna(1).astype(int)

        # 2. Fare Per Person
        X['FarePerPerson'] = X['Fare'] / X['TicketGroupSize']

        # 3. Ticket Prefix
        X['TicketPrefix'] = (
            X['Ticket']
            .str.replace(r'\d', ' ', regex=True)
            .str.replace(r'[./]', ' ', regex=True)
            .str.replace('  ', ' ', regex=True)
            .str.strip()
            .str.upper()
        )
        X['TicketPrefix'] = X['TicketPrefix'].replace('', 'NONE')

        return X

In [11]:
class AgeGroupEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['AgeGroup'] = pd.cut(
            X['Age'],
            bins=[0, 12, 18, 30, 50, 100],
            labels=['Child', 'Teenager', 'YoungAdult', 'Adult', 'Senior']
        )
        return X

In [13]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

trf3 = ColumnTransformer(
    [('ohe', OneHotEncoder(drop='first', sparse_output=False), ['Title', 'AgeGroup', 'TicketPrefix'])],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

In [14]:
num_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex',
            'FamilySize', 'IsAlone', 'Child', 'TicketGroupSize', 'FarePerPerson']

trf4 = ColumnTransformer(
    [('scaling', StandardScaler(), num_cols)],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

In [15]:
from sklearn.pipeline import Pipeline

class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.drop(columns=self.columns, errors='ignore')

pipe = Pipeline([
    ('trans1', trf1),                      # 1. Impute Age
    ('trans2', trf2),                      # 2. Family & Name Features
    ('ticket_eng', TicketEngineer()),      # 3. Ticket Features
    ('age_group', AgeGroupEngineer()),     # 4. Age Binning
    ('trans3', trf3),                      # 5. One Hot Encoding
    ('trans4', trf4),                      # 6. Scaling
    ('drop_cols', DropColumns(['PassengerId', 'Name', 'Ticket', 'Surname'])) # 7. Drop raw text/IDs
])

In [16]:
# Assuming your dataframe is named 'df' and contains the target 'Survived'
# The pipeline will process all columns and leave 'Survived' untouched due to remainder='passthrough'

df_transformed = pipe.fit_transform(df)

# View the final clean dataframe
df_transformed.head()

,Pclass,Age,SibSp,Parch,Fare,Sex,FamilySize,IsAlone,Child,TicketGroupSize,...,TicketPrefix_SO C,TicketPrefix_SOTON O,TicketPrefix_SOTON O Q,TicketPrefix_SOTON OQ,TicketPrefix_STON O,TicketPrefix_SW PP,TicketPrefix_W C,TicketPrefix_W E P,TicketPrefix_WE P,Survived
0,0.827377,-0.565736,0.432793,-0.473674,-0.502445,-0.737695,0.059160,-1.231645,-0.320504,-0.579162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,-1.566107,0.663861,0.432793,-0.473674,0.786845,1.355574,0.059160,-1.231645,-0.320504,-0.579162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.827377,-0.258337,-0.474545,-0.473674,-0.488854,1.355574,-0.560975,0.811922,-0.320504,-0.579162,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1
3,-1.566107,0.433312,0.432793,-0.473674,0.420730,1.355574,0.059160,-1.231645,-0.320504,0.155928,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0.827377,0.433312,-0.474545,-0.473674,-0.486337,-0.737695,-0.560975,0.811922,-0.320504,-0.579162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
